In [0]:
LIST '/Volumes/dbacademy_ecommerce/v01/raw/events-kafka/'

In [0]:
select * from text.`/Volumes/dbacademy_ecommerce/v01/raw/events-kafka`
LIMIT 5;

In [0]:
select * from read_files("/Volumes/dbacademy_ecommerce/v01/raw/events-kafka")
limit 10;

In [0]:
-- using CTAS and read_files() with JSON

DROP TABLE IF EXISTS kafka_events_nronze_raw;

--Create table
create table kafka_events_bronze_raw as
select * from read_files('/Volumes/dbacademy_ecommerce/v01/raw/events-kafka', format => 'json');

-- display the data
select * from kafka_events_bronze_raw
limit 10;

In [0]:
select 
    key as encoded_key,
    cast(unbase64(key) as string) as decoded_key,
    value as encoded_value,
    cast(unbase64(value) as string) as decoded_value
from kafka_events_bronze_raw
limit 5;

In [0]:
create or replace table kafka_events_bronze_decoded as
select 
    cast(unbase64(key) as string) as decoded_key,
    offset,
    partition,
    timestamp,
    topic,
    cast(unbase64(value) as string) as decoded_value
from kafka_events_bronze_raw;

-- View the new table
select * from kafka_events_bronze_decoded
limit 5;

Working with JSON Formatted Strings in a Table

1. Flattening JSON String Columns
    - Query Json Strings

2. Flattening JSON Formatting strings via STRUCT Conversion
    - Converting a Json String to a Column
    - Explode Arrays




In [0]:
select decoded_value,
       decoded_value:device,
        decoded_value:traffic_source,
        decoded_value:geo,
        decoded_value:items
from kafka_events_bronze_decoded
limit 5;

In [0]:
select schema_of_json('{"device":"iOS","ecommerce":{},"event_name":"add_item","event_previous_timestamp":1593880300696751,"event_timestamp":1593880892251310,"geo":{"city":"Westbrook","state":"ME"},"items":[{"item_id":"M_STAN_T","item_name":"Standard Twin Mattress","item_revenue_in_usd":595.0,"price_in_usd":595.0,"quantity":1}],"traffic_source":"google","user_first_touch_timestamp":1593880300696751,"user_id":"UA000000107392458"}') as schema

In [0]:
create or replace table kafka_events_bronze_struct as 
select 
    * except (decoded_value),
    from_json(decoded_value, 'STRUCT<device: STRING, event_name: STRING, event_previous_timestamp: BIGINT, event_timestamp: BIGINT, geo: STRUCT<city: STRING, state: STRING>, items: ARRAY<STRUCT<item_id: STRING, item_name: STRING, item_revenue_in_usd: DOUBLE, price_in_usd: DOUBLE, quantity: BIGINT>>, traffic_source: STRING, user_first_touch_timestamp: BIGINT, user_id: STRING>') as value
from kafka_events_bronze_decoded;

select * from kafka_events_bronze_struct
limit 5;

In [0]:
select decoded_key,
       value.device,
        value.geo.city as city,
        value.items as items,
        array_size(items) as number_elements_in_array
from kafka_events_bronze_struct
order by number_elements_in_array desc
limit 5

In [0]:
-- Exploding items

create or replace table bronze_explode_array as
select 
    decoded_key,
    array_size(value.items) as number_elements_in_array,
    explode(value.items) as item_in_array,
    value.items
from kafka_events_bronze_struct
order by number_elements_in_array desc
limit 5;

-- display table
select * from bronze_explode_array
limit 5

## Variant Column

A **VARIANT** column in Databricks and modern data warehouses stores semi-structured data, such as JSON, XML, or Avro, with flexible schema support. It allows storing nested, complex, or dynamic data types within a single column while retaining query capability.

### Definition
A VARIANT column is a flexible data type capable of holding structured and semi-structured data without strictly enforcing a schema. It is commonly used for ingesting and querying JSON, enabling analytics on nested or schema-evolving data.

### Benefits
- **Schema Flexibility:** Easily stores data with changing or unknown structure, like JSON events.
- **Nested Query Support:** Allows querying and extracting values from nested structures using path syntax.
- **Simplifies Ingestion:** Allows raw ingestion of semi-structured data before schema normalization.
- **Compatibility:** Useful for integrating varied event types or logs without requiring a fixed schema upfront.
- **Enables Analytics:** Supports analytics directly on nested or semi-structured data, making it easier to join, filter, and analyze unstructured sources.



In [0]:
select * from kafka_events_bronze_decoded limit 5;

In [0]:
create or replace table kafka_events_bronze_variant as 
select 
    decoded_key,
    offset,
    partition,
    timestamp,
    topic,
    parse_json(decoded_value) as json_variant_value
from kafka_events_bronze_decoded;

select * from kafka_events_bronze_variant limit 5;

If we wanted to access values use ':' OR if we wanted to enforce schema use '::'

In [0]:
select 
    json_variant_value,
    json_variant_value:device as device,
    json_variant_value:items as items
from kafka_events_bronze_variant
limit 5;

select 
    json_variant_value,
    json_variant_value:device :: string as device,
    json_variant_value:items as items
from kafka_events_bronze_variant
limit 5;